#Introducción

En este ejercicio vamos a desarrollar un grep y wordcount sobre el dataset que elegí en el tema anterior. Esto lo haremos implementando un sistema de ficheros con HDFS para gestionar los ficheros sobre los que haremos el grep y wordcount.

El dataset que utilizaré en esta tarea, es el mismo que utilicé en la tarea de la unidad 1. El origen de este dataset es Kaggle, cuyo dominio es https://www.kaggle.com/datasets/imtkaggleteam/tourism. Este dominio contiene un conjunto de datasets que recogen información acerca del turismo en diferentes paises. En la unidad anterior utilicé de este conjunto de datos los datasets `4- international-tourist-trips.csv`y `11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv`. En este caso utilizaré de nuevo ambos datasets.

##Contextualización del dataset

El dataset `4- international-tourist-trips.csv` recoge información de los paises visitados por turistas a lo largo de los años. En el voy a contar el número de apariciones de cada pais a lo largo de todas las entradas del dataset. Recoge información de 205 paises diferentes. De este modo podré ver cual es el país más representado en nuestro conjunto de datos. Este dataset pesa 128 Kb y contiene un total de 4941 entradas.

El otro dataset que utilizaré es `11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv`. Este dataset recoge información acerca de el número total de personas que trabajan en el sector de la hostelería a lo largo de los años por país. En este el trabajo que haré será el mimso, recoger el número de apariciones que hay de cada pais para ver cuales son los paises con mayor representación en este conjunto de datos. Este dataset pesa 29 Kb y tiene un total de 1046 entradas.

Ambos dataset se relacionan por la columna país o código del país. Podremos ver entre los dos conjuntos de datos.



##Información a extraer con Grep

Voy a utilizar el grep sobre ambos datasets descritos en el apartado anterior; `4- international-tourist-trips.csv` y `11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv`. Voy a contar el número de apariciones de la palabra `Spain` en el conjunto de datos y el número de apariciones de la palabra `France`.

Como sabemos que cada fila solamente contiene un país, sabremos cuantas entradas tienen cada uno de estos dos paises solamente utilizando el comando grep.

##Importación de librerías y descarga de ficheros en Kaggle

Antes de comenzar a trabajar en los puntos descritos a continuación, hay que realizar varias operaciones para definir el marco de trabajo correctamente en el cual vamos a trabajar.

En este apartado voy a importar las librerías necesarias para desempeñar la práctica y descargar los dataset de Kaggle que posteriormente subiré a HDFS y con los cuales realizaré el trabajo.

In [ ]:
import kagglehub #Importo la librería de Kaggle para descargar los ficheros.
import os # Importo la librería necesaria para trabajar con las variables de entorno más adelante.

path = kagglehub.dataset_download("imtkaggleteam/tourism") # Descargo de Kaggle el conjunto de datasets y almaceno la variable donde se ubican

path_tourist = f"{path}/4- international-tourist-trips.csv" # Defino la ruta donde se ubica el dataset de turismo
path_employed = f"{path}/11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv" # Defino la ruta donde se ubica el dataset de empleabilidad

print(f"La ruta donde se ubica el dataset de turismo es: {path_tourist}")
print(f"La ruta donde se ubica el dataset de empleabilidad es: {path_employed}")

100%|██████████| 790k/790k [00:00<00:00, 79.0MB/s]

Extracting files...
La ruta donde se ubica el dataset de turismo es: /root/.cache/kagglehub/datasets/imtkaggleteam/tourism/versions/1/4- international-tourist-trips.csv
La ruta donde se ubica el dataset de empleabilidad es: /root/.cache/kagglehub/datasets/imtkaggleteam/tourism/versions/1/11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv


Hecho esto ya he descargado los archivos y he definido la rutas donde se almacenan los dos dataset que me interesa de cara a subirlos posteriormente al sistema de archivos HDFS.

#1.- Copia de los datasets en el sistema de archivos HDFS

Una vez descargados los datasets voy a comenzar a trabajar por primera vez con Hadoop. Para subir los datasets al sistema de archivos HDFS lo primero que tengo que hacer es establecer el entorno para trabajar con Hadoop.

Voy a ir paso a paso describiendo lo que necesito para instalar Hadoop y dejar el entorno listo para el trabajo.

Como sabemos, Hadoop se basa en Java y es necesario tener instalado Java para poder trabajar con Hadoop. Por ello el primer paso es verificar que Java está instalado y ver que versión tenemos.

In [ ]:
!ls -l /usr/lib/jvm/

total 4
lrwxrwxrwx 1 root root   21 Oct 23 13:09 java-1.17.0-openjdk-amd64 -> java-17-openjdk-amd64
drwxr-xr-x 9 root root 4096 Dec 11 14:13 java-17-openjdk-amd64


Como podemos ver, Java está correctamente instalado cib ka versión 17. En caso de no tenerlo instalado habría que descargar e instalar la versión que nos interese.

Una vez verificada la instalación de Java ya puedo comenzar a instalar Hadoop.

Voy a instalar la versión 3.4.2 de Apache Hadoop, ya que a día de hoy es la última versión según podemos ver en su página oficial (https://hadoop.apache.org/releases.html) y es la misma que se está utilizando en la teoría del temario.

Descargaré Hadoop con el comando wget definiendo la ruta de la versión que me interesa (`https://downloads.apache.org/hadoop/common/hadoop-3.4.2/hadoop-3.4.2.tar.gz`).

In [ ]:
!wget https://downloads.apache.org/hadoop/common/hadoop-3.4.2/hadoop-3.4.2.tar.gz

--2026-01-13 14:37:58--  https://downloads.apache.org/hadoop/common/hadoop-3.4.2/hadoop-3.4.2.tar.gz
Resolving downloads.apache.org (downloads.apache.org)... 88.99.208.237, 135.181.214.104, 2a01:4f9:3a:2c57::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|88.99.208.237|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1065831750 (1016M) [application/x-gzip]
Saving to: ‘hadoop-3.4.2.tar.gz’

hadoop-3.4.2.tar.gz 100%[===================>]   1016M  25.5MB/s    in 42s     

2026-01-13 14:38:40 (24.4 MB/s) - ‘hadoop-3.4.2.tar.gz’ saved [1065831750/1065831750]



Una vez descargada vamos a verificar que se ha descargado correctamente.

In [ ]:
!ls

hadoop-3.4.2.tar.gz  sample_data


Como podemos ver con el comando `ls` ya tenemos nuestra distribución descargada correctamente con el nombre `hadoop-3.4.2.tar.gz`.

Si nos fijamos la extensión del fichero es `.gz` esto quiere decir que es un archivo comprimido, así que el siguiente paso es descomprimir el archivo utilizando el comando `tar`.

In [ ]:
!tar -xzf hadoop-3.4.2.tar.gz
!ls -l

total 1040864
drwxr-xr-x 10 1001 1001       4096 Aug 20 11:13 hadoop-3.4.2
-rw-r--r--  1 root root 1065831750 Oct  4 09:30 hadoop-3.4.2.tar.gz
drwxr-xr-x  1 root root       4096 Dec 11 14:34 sample_data


Tras unos segundos se ha descomprimido la distribución de Hadoop en una carpeta llamada `hadoop-3.4.2`. Ahora voy a mover esta carpeta a `/usr/local/hadoop` para ubicarla en la ruta donde se suelen utilizar las aplicaciones del usuario.

In [ ]:
!mv  hadoop-3.4.2/ /usr/local/hadoop

In [ ]:
!ls /usr/local/hadoop

bin  include  libexec	      licenses-binary  NOTICE-binary  README.txt  share
etc  lib      LICENSE-binary  LICENSE.txt      NOTICE.txt     sbin


Como podemos verificar ya hemos movido correctamente la carpeta descomprimida. Así que podemos considerar que ya hemos instalado correctamente Hadoop en nuestro sistema.

Pero con esto no está todo el trabajo hecho, ahora voy a actualizar las variables de entorno `Java_HOME`, `PATH` y `HADOOP_HOME`. Aunque este no es un paso obligatorio si nos facilita mucho el trabajo a la hora de desarrollar en el resto de la práctica.

Si no actualizase estas variables de entorno, cada vez que quisiese utilizar hadoop o java, tendría que definir en la linea de comandos la ruta completa donde está el fichero java o hadoop que voy a ejecutar. De esta forma no me hará falta definir la ruta entera en cada comando, porque estoy definiendo el alias para llamar Java y Hadoop.

In [ ]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64" # Defino el alias para usar java apuntando a la distribución de java instalada
os.environ["HADOOP_HOME"] = "/usr/local/hadoop" # Defino el alias para utilizar hadoop apuntando a la carpeta donde hemos movido la distribución de hadoop descargada
os.environ["PATH"] += f":{os.environ['HADOOP_HOME']}/bin:{os.environ['HADOOP_HOME']}/sbin" # Actualizo la variable de entorno PATH a la carpeta de Hadoop para poder utilizar sus comandos desde cualquier lugar


Para verificar que se ha instalado y actualizado el alias correctamente, voy a verificar la instalación de hadoop con el comando `hadoop version`.

In [ ]:
!hadoop version

Hadoop 3.4.2
Source code repository https://github.com/apache/hadoop.git -r 84e8b89ee2ebe6923691205b9e171badde7a495c
Compiled by ahmarsu on 2025-08-20T10:30Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4
From source with checksum fa94c67d4b4be021b9e9515c9b0f7b6
This command was run using /usr/local/hadoop/share/hadoop/common/hadoop-common-3.4.2.jar


Como podemos comprobar ya está el entorno con hadoop instalado y configurado listo para trabajar.

Aunque he ubicado esta parte en el ejercicio 1, podía haberse ubicado en la introducción también. He decidido ubicarlo en este apartado ya que era el primer lugar donde iba a requerir la instalación de hadoop.

Con la instalación de Hadoop tenemos ya disponibles muchas herramientas y comandos, el que nos interesa en este punto es HDFS. Podemos obtener una lista de comandos disponibles usando `hdfs --help`.

In [ ]:
!hdfs --help

Usage: hdfs [OPTIONS] SUBCOMMAND [SUBCOMMAND OPTIONS]

  OPTIONS is none or any of:

--buildpaths                       attempt to add class files from build tree
--config dir                       Hadoop config directory
--daemon (start|status|stop)       operate on a daemon
--debug                            turn on shell script debug mode
--help                             usage information
--hostnames list[,of,host,names]   hosts to use in worker mode
--hosts filename                   list of hosts to use in worker mode
--loglevel level                   set the log4j level for this command
--workers                          turn on worker mode

  SUBCOMMAND is one of:


    Admin Commands:

cacheadmin           configure the HDFS cache
crypto               configure HDFS encryption zones
debug                run a Debug Admin to execute HDFS debug commands
dfsadmin             run a DFS admin client
dfsrouteradmin       manage Router-based federation
ec                   run a HD

Vemos que en efecto el comando para trabajar con hdfs funciona correctamente. Si nos fijamos en la ayuda que nos ha mostrado el comando, lo que nos permite trabajar con el sistema de archivos de hadoop es el comando `hdfs dfs`. Este comando ofrece bastantes herramientas para trabajar con ficheros y directorios al igual que linux.

Ahora que tenemos todo listo, vamos a subir los archivos del dataset que vamos a utilizar al gestor de archivos HDFS. Es importante remarcar que los archivos que tenemos en la máquina localo no son los archivos que hay en HDFS. Son "entidades" independientes, ya que HDFS es un gestor de archivos distribuido, hay que tener especial cuidado siendo conscientes de cuando se trabaja con un archivo en local o en HDFS.

Ahora tenemos los archivos `.csv` desgargados de Kaggle únicamente en local. En este primer punto vamos a subirlos a HDFS. Para ello se utiliza la instrucción `put`.

Antes de subirlos vamos a verificar los archivos que hay en HDFS para verificar que no hay nada y posteriormente poder apreciar como se suben correctamente.

In [ ]:
!hdfs dfs -ls

Found 3 items
drwxr-xr-x   - root root       4096 2025-12-11 14:34 .config
-rw-r--r--   1 root root 1065831750 2025-10-04 09:30 hadoop-3.4.2.tar.gz
drwxr-xr-x   - root root       4096 2025-12-11 14:34 sample_data


Vemos como en efecto no están los archivos en HDFS, ahora voy a subirlos. Como hemos visto cuando he descargado de Kaggle los ficheros las rutas de estos son:
*   `/kaggle/input/tourism/4- international-tourist-trips.csv`
*   `/kaggle/input/tourism/11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv`

Voy a subirlos utilizando el comando `hdfs dfs -put`.

___
*NOTA: Cuando descargas de kaggle con su api los datasets te devuelve la ruta donde se han almacenado, lo he hecho al principio del cuadernillo, pero esta llamada tiene una particularidad, la primera vez que se ejecuta los guarda en una ruta, pero el resto de veces la ruta es diferente. Esto lo hace para cachearlos y agilizar las descargas posteriores de ficheros ya existentes.

Como esto puede originar que según la ruta que utilice y las veces que se haya ejecutado el cuadernillo haya ocasiones que no se suban con las rutas que he indicado, voy a hacer el put en hdfs utilizando dos rutas por cada fichero. Esto no causará problema pues con la opción -f en caso de que el fichero exista lo pisará. En el peor de los casos subirá dos veces el mismo fichero (aunque solo quedará uno) y en ocasiones puede que veamos que un comando ha fallado al no encontrar el fichero.
___

In [ ]:
!hdfs dfs -put -f "/kaggle/input/tourism/4- international-tourist-trips.csv"
!hdfs dfs -put -f "/kaggle/input/tourism/11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv"
!hdfs dfs -put -f "/root/.cache/kagglehub/datasets/imtkaggleteam/tourism/versions/1/4- international-tourist-trips.csv"
!hdfs dfs -put -f "/root/.cache/kagglehub/datasets/imtkaggleteam/tourism/versions/1/11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv"


put: `/kaggle/input/tourism/4- international-tourist-trips.csv': No such file or directory
put: `/kaggle/input/tourism/11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv': No such file or directory


*Ver la nota informativa del cuadro de texto anterior. Si el mensaje muestra que `No such file or directory` es porque la ejecución es la primera. Se habrá subido correctamente con el segundo put del mismo fichero.

Parece haber ido todo correctamente. Si nos fijamos en el comando he utilizado la opción `-f` para pisar el archivo en caso de que exista ya con el nuevo que intento subir. De esta forma me aseguro que siempre permanece lo último que intento subir y no me dará error si el fichero ya existe.

Voy a verificar que realmente se han subido bien.

In [ ]:
!hdfs dfs -ls

Found 5 items
drwxr-xr-x   - root root       4096 2025-12-11 14:34 .config
-rw-r--r--   1 root root      29619 2026-01-13 14:39 11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv
-rw-r--r--   1 root root     130440 2026-01-13 14:39 4- international-tourist-trips.csv
-rw-r--r--   1 root root 1065831750 2025-10-04 09:30 hadoop-3.4.2.tar.gz
drwxr-xr-x   - root root       4096 2025-12-11 14:34 sample_data


Se puede ver que los dos archivos se han subido correctamente, para trabajar más facilmente con ellos voy a crear una carpeta llamada datasets y los voy a meter en ella.

In [ ]:
!hdfs dfs -mkdir datasets
!hdfs dfs -mv "11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv" datasets
!hdfs dfs -mv "4- international-tourist-trips.csv" datasets

Voy a verificar que ya no están en la raíz.

In [ ]:
!hdfs dfs -ls

Found 4 items
drwxr-xr-x   - root root       4096 2025-12-11 14:34 .config
drwxr-xr-x   - root root       4096 2026-01-13 14:39 datasets
-rw-r--r--   1 root root 1065831750 2025-10-04 09:30 hadoop-3.4.2.tar.gz
drwxr-xr-x   - root root       4096 2025-12-11 14:34 sample_data


Podemos ver que han desaparecido de la raiz y que en su lugar hay una carpeta llamada `datasets`. Ahora voy a verificar que sí están en la carpeta objetivo.

In [ ]:
!hdfs dfs -ls datasets/

Found 2 items
-rw-r--r--   1 root root      29619 2026-01-13 14:39 datasets/11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv
-rw-r--r--   1 root root     130440 2026-01-13 14:39 datasets/4- international-tourist-trips.csv


En efecto ahora ya están donde me interesaban, ya he aislado los ficheros para que el trabajo posterior sea más limpio y se realice todo exclusivamente en esta carpeta.

#2.- Conteo mediante grep

##Contextualización grep linux

En este punto voy a contar mediante el comando grep el número de apariciones de la palabra Spain y el número de apariciones de la palabra France en los dos ficheros que hemos subido a HDFS en el punto anterior.

Grep es un comando utilizado en sistemas Linux que te permite buscar dada un texto la palabra o expresión regular indicada. Por ejemplo, si utilizamos ls para listar el contenido de una carpeta, podemos concatenarlo con grep para ver cuantos de los archivos contienen la palabra o expresión regular indicada.

Por ejemplo, en nuestro caso tenemos dos ficheros en la carpeta datasets; `4- international-tourist-trips.csv` y `11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv`. Si utilizamos el comando `hdfs dfs -ls datasets/ | grep .csv` nos mostrará todos los archivos que contengan en el nombre `.csv`, como ambos lo tienen nos listará los dos archivos, sin embargo si usamos `hdfs dfs -ls datasets/ | grep 4-` nos mostrará todos los archivos que tengan en el nombre `4-`, en este caso solo el primero.

Voy a probar lo que acabo de señalar.

In [ ]:
!hdfs dfs -ls datasets/ | grep .csv

-rw-r--r--   1 root root      29619 2026-01-13 14:39 datasets/11- number-of-people-employed-in-food-and-beverage-serving-activities-per-1000-population.csv
-rw-r--r--   1 root root     130440 2026-01-13 14:39 datasets/4- international-tourist-trips.csv


In [ ]:
!hdfs dfs -ls datasets/ | grep 4-

-rw-r--r--   1 root root     130440 2026-01-13 14:39 datasets/4- international-tourist-trips.csv


Vemos como en la primera prueba nos lista los dos archivos, y en la segunda solamente nos ha listado uno, el que contenía la ocurrencia que hemos definido. En estos ejemplos he utilizado palabras, pero se podrían utilizar expresiones regulares en su lugar.


##Grep MapReduce

Esta pequeña introducción que he hecho es para contextualizar el comando grep de linux, aunque no es exáctamente lo que vamos a hacer en este punto. Hadoop tiene un conjunto de herramientas o programas por defecto que aportan funcionalidades ejemplificando el trabajo con ficheros en el sistema distribuido HDFS a través de MapReduce. Una de estas funcionalidades es grep, que hace algo similar a lo mostrado en el ejemplo, pero indicando el número de apariciones de la palabra buscada.

La función grep se ubica en el archivo java descargado por defecto en la instalación de hadoop `hadoop-mapreduce-examples-3.4.2.jar`. Este programa java contiene varias opciones además del grep, las cuales están listadas en la teoría.

En este caso, como he indicado al principio del ejercicio quiero buscar el número de apariciones de la palabra Spain y France en nuestros datasets que hemos subido a HDFS. Se puede buscar en un fichero determinado o en todos los ficheros de una carpeta. Yo buscaré en ambos ficheros de la carpeta `datasets` simultaneamente.

Para hacer esta búsqueda voy a utilizar los comandos:
* !hadoop jar /usr/local/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.2.jar grep datasets grep_spain 'Spain'
* !hadoop jar /usr/local/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.2.jar grep datasets grep_france 'France'

En ellos lo que hago es utilizar el archivo con funcionalidades de ejemplo de MapReduce `hadoop-mapreduce-examples-3.4.2.jar` que está ubicado en `/usr/local/hadoop/share/hadoop/mapreduce/` e indicarle que quiero utilizar la función `grep`. Posteriormente le digo sobre que carpeta quiero buscar y como quiero que llame la carpeta en la que va a almacenar los resultados de la búsqueda. Por último le indico la expresión regular o palabra que quiero que busque en el fichero o conjunto de ficheros.

En el primer caso le estoy diciendo, ejecuta grep sobre la carpeta datasets, déjame los resultados en la caarpeta `grep_spain` y busca la palabra `Spain`. En el segundo caso hago lo mismo pero con la palabra `France` y su carpeta correspondiente `grep_france`.

In [ ]:
!hadoop jar /usr/local/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.2.jar grep datasets grep_spain 'Spain' # Ejecutamos la aplicación de ejemplos de java y pasamos como parámetros grep (función a utilizar) sobre la carpeta
!hadoop jar /usr/local/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.2.jar grep datasets grep_france 'France'

2026-01-13 14:39:33,088 INFO input.FileInputFormat: Total input files to process : 2
2026-01-13 14:39:33,124 INFO mapreduce.JobSubmitter: number of splits:2
2026-01-13 14:39:33,602 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1488327849_0001
2026-01-13 14:39:33,603 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-01-13 14:39:33,863 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-01-13 14:39:33,864 INFO mapreduce.Job: Running job: job_local1488327849_0001
2026-01-13 14:39:33,866 INFO mapred.LocalJobRunner: OutputCommitter set in config null
2026-01-13 14:39:33,879 INFO output.PathOutputCommitterFactory: No output committer factory defined, defaulting to FileOutputCommitterFactory
2026-01-13 14:39:33,880 INFO output.FileOutputCommitter: File Output Committer Algorithm version is 2
2026-01-13 14:39:33,881 INFO output.FileOutputCommitter: FileOutputCommitter skip cleanup _temporary folders under output directory:false, ignore cleanup

Una vez he ejecutado la aplicación con el grep, voy a ver si realmente lo ha hecho bien, por la salida mostrada por consola tiene buena pinta.

____
*NOTA INFORMATIVA: Aunque con esta instalación de hadoop coincide la carpeta de trabajo con la raíz del sistema HDFS montado, podría utilizar los comandos de linux standar (ls, cat, mv, etc...) pero como el objetivo de la práctica es trabajar con el sistema de archivos distribuido HDFS voy a utilizar los comandos a través de hdfs aunque sea algo más lento y engorroso para estos casos con ficheros pequeños. Pero en estos casos utilizar el comando `ls` y `hdfs dfs -ls` devolverían los mismos resultados.
____

In [ ]:
!hdfs dfs -ls

Found 6 items
drwxr-xr-x   - root root       4096 2025-12-11 14:34 .config
drwxr-xr-x   - root root       4096 2026-01-13 14:39 datasets
drwxr-xr-x   - root root       4096 2026-01-13 14:39 grep_france
drwxr-xr-x   - root root       4096 2026-01-13 14:39 grep_spain
-rw-r--r--   1 root root 1065831750 2025-10-04 09:30 hadoop-3.4.2.tar.gz
drwxr-xr-x   - root root       4096 2025-12-11 14:34 sample_data


Vemos que me ha creado correctamente las dos carpetas para las búsquedas realizadas. Una para la búsqueda de la palabra `Spain` -> `grep_spain` y otra para la palabra `France` -> `grep_france`.

Ahora voy a ver el contenido de estas carpetas y los ficheros alojados en ellas.

In [ ]:
!hdfs dfs -ls grep_spain/
!hdfs dfs -ls grep_france/

Found 2 items
-rw-r--r--   1 root root          0 2026-01-13 14:39 grep_spain/_SUCCESS
-rw-r--r--   1 root root          9 2026-01-13 14:39 grep_spain/part-r-00000
Found 2 items
-rw-r--r--   1 root root          0 2026-01-13 14:39 grep_france/_SUCCESS
-rw-r--r--   1 root root         10 2026-01-13 14:39 grep_france/part-r-00000


Vemos que cada carpeta contiene dos ficheros con los nombres idénticos. El primero llamado `part-r-00000` el cual contendrá el resultado de la búsqueda y el segundo `_SUCCESS` que indica que la tarea se ha realizado correctamente.

Voy a explorar con el comando `cat` el contenido de ambos ficheros para ver el número de apariciones de cada palabra.

In [ ]:
!hdfs dfs -cat grep_spain/part-r-00000

53	Spain


In [ ]:
!hdfs dfs -cat grep_france/part-r-00000

42	France


Como podemos ver la búsqueda por ambas palabras ha arrojado resultados, en ambos casos ha encontrado ocurrencias. Vemos que ha encontrado 53 ocurrencias de la palabra Spain entre los dos datasets y 42 ocurrencias de la palabra France.

Con esta búsqueda podemos concluir que hay más representación de registros de turismo y empleo en hostelería de España que de Francia, no es un volumen muy superior, porque entre los dos datasets la diferencia es de 11 resultados.

Esta misma búsqueda se podría realizar sobre un fichero en particular indicando el nombre del fichero, o incluso entre muchos más ficheros.

#3.- Contar apariciones de cada palabra con MapReduce en Java

En el punto anterior he contado las apariciones de las palabras Spain y France utilizando los ejemplos proporcionados de MapReduce. En este punto voy a implementar en una aplicación java mi propio MapReduce que se encargará de contar las distintas palabras que aparecen en la columna `Entity` referente al pais de mis ficheros .csv que he subido al sistema de archivos HDFS.

MapReduce se encarga de la gran parte de trabajo sobre la gestión de archivos grandes, tanto de dividirlos y paralelizar las tareas para optimizar las funciones que apliquemos en nuestro programa java. Por ello el programa Java que desarrolle en este punto solo tiene que implementar las funciones Map y Reduce, del resto se encarga el framework solito.

Mi archivo java se va a llamar `CountryCounter.java` y voy a ir desarrollándolo en Colab. Voy a escribir el código y posteriormente lo explicaré más en detalle.

## Definición y creación de la clase

In [ ]:
%%writefile -a CountryCounter.java
import java.io.IOException;
import java.util.StringTokenizer;

import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Job;
import org.apache.hadoop.mapreduce.Mapper;
import org.apache.hadoop.mapreduce.Reducer;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;

public class CountryCounter {

	public static class IncreaseCountryCounterMapper extends Mapper<Object, Text, Text, IntWritable>{

		// MapReduce invocará a este método una vez por cada línea del fichero
		public void map(Object key, Text value, Context context) throws IOException, InterruptedException {

			// Cogemos la línea que llega como parámetro, la convertimos a String
			// y la dividimos en los distintos bloques de información
			String valueString = value.toString();
			String[] dataOfTourism = valueString.split(",");

			// Obtengo el nombre del pais (primer elemento del índice)
			String countryName = dataOfTourism[0];


			// Ya que mi dataset tiene cabecera, filtro para que no cuente el título de la columna como país
			if (!countryName.equals("Entity"))
				context.write(new Text(countryName), new IntWritable(1));
			}
		}



	public static class IntSumReducer extends Reducer<Text,IntWritable,Text,IntWritable> {

		private IntWritable result = new IntWritable();

		//Se invocará este método por cada país
		public void reduce(Text key, Iterable<IntWritable> values, Context context) throws IOException, InterruptedException {
			int sum = 0;

			// Se suma el número de apariciones de cada país al total
			for (IntWritable val : values) {
				sum += val.get();
			}
			result.set(sum);
			context.write(key, result);
		}
	}


	public static void main(String[] args) throws Exception {
		Configuration conf = new Configuration();
		Job job = Job.getInstance(conf, "CountryCounter");
		job.setJarByClass(CountryCounter.class);
		job.setMapperClass(IncreaseCountryCounterMapper.class);
		job.setCombinerClass(IntSumReducer.class);
		job.setReducerClass(IntSumReducer.class);
		job.setOutputKeyClass(Text.class);
		job.setOutputValueClass(IntWritable.class);
		FileInputFormat.addInputPath(job, new Path(args[0]));
		FileOutputFormat.setOutputPath(job, new Path(args[1]));
		System.exit(job.waitForCompletion(true) ? 0 : 1);
	}
}

Writing CountryCounter.java


## Explicación de la clase

La clase `CountryCounter` contiene dos clases. La primera es `IncreaseCountryCounterMapper` que extiende la clase `Mapper` y contendrá el método `map` con nuestra lógica. La segunda clase es `IntSumReducer` la cual extenderá a la clase `Reducer` e implementará el método `reduce` con la lógica.

###Imports

Primero irá la sección de los import y la declaración de la clase `IncreaseCountryCounterMapper`, en esta sección se agregan los import que serán necesarios para trabajar con las librerías.

```
import java.io.IOException;
import java.util.StringTokenizer;

import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Job;
import org.apache.hadoop.mapreduce.Mapper;
import org.apache.hadoop.mapreduce.Reducer;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;

public class CountryCounter {
```

### Mapper

El mapper contiene el método map, el cual se invocará por cada linea de nuestro csv. Vamos a trabajar con dos csvs. El primero sigue el formato Pais,Codigo_pais,Año,Turistas y el segundo tiene el formato País,Codigo_pais,Año,Empleados_hostelería. El objetivo de mi código es contar cuantas apariciones hay de cadad país.

Como podemos ver tengo la suerte de que ambos csv separan sus elementos por comas y el campo referente al país (`Entity`) es el que se ubica en primera posición. Así que aunque no es lo común, el mismo método map funcionará para ambos csvs. Como solamente quiero saber cuantas apariciones hay por cada país, me basta con obtener la línea, transformarla en un string y separo los elementos de esta linea por el caracter ','. Una vez tengo el array de elementos, como solo me interesa el primero y me interesa íntegramente, pues contiene exclusivamente el nombre del país y es lo que quiero, obtengo el primer elemento del array. El resto de elementos los desecho pues no me interesa. Como la primera linea de estos csvs contiene el nombre de cada columna, he agregado una condición para que verifique que el texto es diferente a `Entity` nombre de la columna que contiene el país. Así evito que considere el nombre de la columna como si fuese un país. En caso de que sea diferente de Entity, lo agrego al contexto y le añado el valor 1, pues cada linea como mucho tendrá un país.

```
public static class IncreaseCountryCounterMapper extends Mapper<Object, Text, Text, IntWritable>{

		// MapReduce invocará a este método una vez por cada línea del fichero
		public void map(Object key, Text value, Context context) throws IOException, InterruptedException {

			// Cogemos la línea que llega como parámetro, la convertimos a String
			// y la dividimos en los distintos bloques de información
			String valueString = value.toString();
			String[] dataOfTourism = valueString.split(",");

			// Obtengo el nombre del pais (primer elemento del índice)
			String countryName = dataOfTourism[0];


			// Ya que mi dataset tiene cabecera, filtro para que no cuente el título de la columna como país
			if (!countryName.equals("Entity"))
				context.write(new Text(SingleCountryData[countryName]), new IntWritable(1));
			}
		}
	}
```


### Reducer

Una vez hecho el mapper, el reducer es más sencillo, extenderá la clase Reducer y tendrá su método `reduce` con la lógica. Este método será invocado una vez por cada país.

Primero se define la variable que contendrá el número de apariciones del país llamada `sum`. Nos recorremos todos los valores de cada país, y obtenemos su valor, el cual hemos definido en el maper y lo agregamos a nuestro contador. Como el valor solamente será 1 pues no puede haber más de una aparición por linea podríamos haber hecho `sum ++;` pero está mejor hecho obteniendo su valor y sumándolo, es lo que habría que hacer si el número no siempre fuese uno.

Una vez recorridas y contadas todas las apariciones de un pais se agrega el valor al contexto para que MapReduce siga con su trabajo.

```
public static class IntSumReducer extends Reducer<Text,IntWritable,Text,IntWritable> {

		private IntWritable result = new IntWritable();

		//Se invocará este método por cada país
		public void reduce(Text key, Iterable values, Context context) throws IOException, InterruptedException {
			int sum = 0;

			// Se suma el número de apariciones de cada país al total
			for (IntWritable val : values) {
				sum += val.get();
			}
			result.set(sum);
			context.write(key, result);
		}
	}
```

### Main

Este método es prácticamente igual siempre, en el definimos el método `main`. Es el encargado de decirle a java que es lo que tiene que ejecutar y con que condiciones, es el primero que se invoca al llamar al programa.

En el se define la configuración del MapReduce. Hay que tener cuidado con invocara la clase correcta y definir el Mapper y Reduccer que queremos aplicar.

```
public static void main(String[] args) throws Exception {
		Configuration conf = new Configuration();
		Job job = Job.getInstance(conf, "CountryCounter");
		job.setJarByClass(CountryCounter.class);
		job.setMapperClass(IncreaseCountryCounterMapper.class);
		job.setCombinerClass(IntSumReducer.class);
		job.setReducerClass(IntSumReducer.class);
		job.setOutputKeyClass(Text.class);
		job.setOutputValueClass(IntWritable.class);
		FileInputFormat.addInputPath(job, new Path(args[0]));
		FileOutputFormat.setOutputPath(job, new Path(args[1]));
		System.exit(job.waitForCompletion(true) ? 0 : 1);
	}
}
```

Hecho esto nuestra clase java está lista.

## Compilación de la clase

In [ ]:
!hdfs dfs -ls

Found 7 items
drwxr-xr-x   - root root       4096 2025-12-11 14:34 .config
-rw-r--r--   1 root root       2443 2026-01-13 14:39 CountryCounter.java
drwxr-xr-x   - root root       4096 2026-01-13 14:39 datasets
drwxr-xr-x   - root root       4096 2026-01-13 14:39 grep_france
drwxr-xr-x   - root root       4096 2026-01-13 14:39 grep_spain
-rw-r--r--   1 root root 1065831750 2025-10-04 09:30 hadoop-3.4.2.tar.gz
drwxr-xr-x   - root root       4096 2025-12-11 14:34 sample_data


He verificado que el fichero se ha creado correctamente. Con nuestro fichero java listo, solo nos queda compilarlo y generar el jar para ejecutarlo y que cuente las apariciones de los paises.

Para compilarlo utilizaré el comando `javac` al cual habrá que indicarle la ruta de las dependencias de Hadoop. Esto nos generará a partir del fichero .java nuestro .class. Después generaré el paquete jar ejecutable por Hadoop con el comando `jar`.

Las instrucciones finales serán:

1.   `javac -cp /usr/local/hadoop/share/hadoop/common/*:/usr/local/hadoop/share/hadoop/mapreduce/* CountryCounter.java -Xlint`

2.   `jar -cvf countrycounter.jar *.class`




In [ ]:
!javac -cp /usr/local/hadoop/share/hadoop/common/*:/usr/local/hadoop/share/hadoop/mapreduce/* CountryCounter.java -Xlint
!jar -cvf countrycounter.jar *.class

/usr/local/hadoop/share/hadoop/common/hadoop-common-3.4.2.jar(/org/apache/hadoop/fs/Path.class): warning: Cannot find annotation method 'value()' in type 'LimitedPrivate': class file for org.apache.hadoop.classification.InterfaceAudience not found
1 warning
added manifest
adding: CountryCounter$IncreaseCountryCounterMapper.class(in = 1678) (out= 702)(deflated 58%)
adding: CountryCounter$IntSumReducer.class(in = 1770) (out= 753)(deflated 57%)
adding: CountryCounter.class(in = 1544) (out= 828)(deflated 46%)


In [ ]:
!hdfs dfs -ls

Found 11 items
drwxr-xr-x   - root root       4096 2025-12-11 14:34 .config
-rw-r--r--   1 root root       1678 2026-01-13 14:39 CountryCounter$IncreaseCountryCounterMapper.class
-rw-r--r--   1 root root       1770 2026-01-13 14:39 CountryCounter$IntSumReducer.class
-rw-r--r--   1 root root       1544 2026-01-13 14:39 CountryCounter.class
-rw-r--r--   1 root root       2443 2026-01-13 14:39 CountryCounter.java
-rw-r--r--   1 root root       3095 2026-01-13 14:39 countrycounter.jar
drwxr-xr-x   - root root       4096 2026-01-13 14:39 datasets
drwxr-xr-x   - root root       4096 2026-01-13 14:39 grep_france
drwxr-xr-x   - root root       4096 2026-01-13 14:39 grep_spain
-rw-r--r--   1 root root 1065831750 2025-10-04 09:30 hadoop-3.4.2.tar.gz
drwxr-xr-x   - root root       4096 2025-12-11 14:34 sample_data


Como podemos ver se ha generado correctamente nuestro .jar `countrycounter.jar`. Ya tenemos lista la aplicación para contar el número de apariciones de los paises con Hadoop.

## Ejecución del MapReducer

Para contarlos habrá que ejecutar la instrucción `hadoop jar countrycounter.jar CountryCounter datasets/ CountryCounter`. Esta instrucción se compone de varias partes. Con `hadoop jar countrycounter.jar` indicamos el fichero jar que queremos ejecutar con Hadoop, el siguiente parámetro `CountryCounter` indica cual es la clase principal que tiene el Mapper y el Reducer. Los siguientes dos parámetros indican el input y el output. En mi caso el input es `datasets/` pues quiero que se ejecute sobre todos los ficheros de la carpeta datasets y el output es `CountryCounter` porque es la carpeta que se va a crear con los resultados de la ejecución del MapReduce.

In [ ]:
!hadoop jar countrycounter.jar CountryCounter datasets/ CountryCounter

2026-01-13 14:40:00,668 WARN mapreduce.JobResourceUploader: Hadoop command-line option parsing not performed. Implement the Tool interface and execute your application with ToolRunner to remedy this.
2026-01-13 14:40:00,966 INFO input.FileInputFormat: Total input files to process : 2
2026-01-13 14:40:01,062 INFO mapreduce.JobSubmitter: number of splits:2
2026-01-13 14:40:01,645 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local1231985719_0001
2026-01-13 14:40:01,647 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-01-13 14:40:02,150 INFO mapreduce.Job: The url to track the job: http://localhost:8080/
2026-01-13 14:40:02,151 INFO mapreduce.Job: Running job: job_local1231985719_0001
2026-01-13 14:40:02,155 INFO mapred.LocalJobRunner: OutputCommitter set in config null
2026-01-13 14:40:02,174 INFO output.PathOutputCommitterFactory: No output committer factory defined, defaulting to FileOutputCommitterFactory
2026-01-13 14:40:02,175 INFO output.FileOutputCommitter

Una vez ejecutado el código fijándome en los logs que pinta parece que ha ido bien. Pero para asegurarnos voy a hacer un ls y debería haber una carpeta llamada `CountryCounter`.

In [ ]:
!hdfs dfs -ls

Found 12 items
drwxr-xr-x   - root root       4096 2025-12-11 14:34 .config
drwxr-xr-x   - root root       4096 2026-01-13 14:40 CountryCounter
-rw-r--r--   1 root root       1678 2026-01-13 14:39 CountryCounter$IncreaseCountryCounterMapper.class
-rw-r--r--   1 root root       1770 2026-01-13 14:39 CountryCounter$IntSumReducer.class
-rw-r--r--   1 root root       1544 2026-01-13 14:39 CountryCounter.class
-rw-r--r--   1 root root       2443 2026-01-13 14:39 CountryCounter.java
-rw-r--r--   1 root root       3095 2026-01-13 14:39 countrycounter.jar
drwxr-xr-x   - root root       4096 2026-01-13 14:39 datasets
drwxr-xr-x   - root root       4096 2026-01-13 14:39 grep_france
drwxr-xr-x   - root root       4096 2026-01-13 14:39 grep_spain
-rw-r--r--   1 root root 1065831750 2025-10-04 09:30 hadoop-3.4.2.tar.gz
drwxr-xr-x   - root root       4096 2025-12-11 14:34 sample_data


Vemos que la carpeta existe, ahora vamos a listar los ficheros que contiene.

In [ ]:
!hdfs dfs -ls CountryCounter/

Found 2 items
-rw-r--r--   1 root root          0 2026-01-13 14:40 CountryCounter/_SUCCESS
-rw-r--r--   1 root root       2733 2026-01-13 14:40 CountryCounter/part-r-00000


Vemos que tiene dos ficheros; `_SUCCESS` y `part-r-00000`, como nos ocurrió con el ejemplo del grep, el primero nos indica que la ejecución se ha realizado con éxito y el segundo contiene los resultados del MapReduce. Ahora voy a realizar un cat del fichero `part-r-00000` para ver cuales han sido los resultados.

In [ ]:
!hdfs dfs -cat CountryCounter/part-r-00000

Albania	15
American Samoa	23
Andorra	23
Angola	42
Anguilla	27
Antigua and Barbuda	27
Argentina	36
Armenia	27
Aruba	27
Australia	26
Austria	37
Azerbaijan	19
Bahamas	27
Bahrain	30
Bangladesh	27
Barbados	27
Belarus	24
Belgium	29
Belize	34
Benin	26
Bermuda	27
Bhutan	27
Bolivia	27
Bonaire Sint Eustatius and Saba	16
Bosnia and Herzegovina	36
Botswana	24
Brazil	42
British Virgin Islands	27
Brunei	18
Bulgaria	27
Burkina Faso	27
Burundi	23
Cambodia	27
Cameroon	20
Canada	27
Cape Verde	27
Cayman Islands	27
Central African Republic	26
Chad	27
Chile	40
China	26
Colombia	27
Comoros	30
Congo	30
Cook Islands	27
Costa Rica	36
Cote d'Ivoire	4
Croatia	43
Cuba	27
Curacao	27
Cyprus	42
Czechia	44
Democratic Republic of Congo	22
Denmark	42
Djibouti	25
Dominica	27
Dominican Republic	32
East Timor	16
Ecuador	27
Egypt	31
El Salvador	42
Estonia	43
Eswatini	29
Ethiopia	27
Fiji	27
Finland	38
France	42
French Guiana	10
French Polynesia	40
Gabon	11
Gambia	27
Georgia	30
Germany	39
Ghana	27
Greece	47
Grenada	27
Guadel

Con los resultados del cat, ya podemos confirmar que el MapReduce ha hecho justo lo que pretendía, contar el número de apariciones de cada país en nuestro dataset e indicarlo.

Al ver los resultados, se puede apreciar como MapReduce hace mucho más que nuestra implementación del Map y del Reduce. Ha agrupado todas las ocurrencias de los paises que hemos detectado en el map y las ha ordenado alfabéticamente.

#4.- Contar apariciones de cada palabra con MapReduce usando Python con Hadoop-Streaming

En este ejercicio voy a implementar la misma lógica de MapReduce que he aplicado en el ejercio anterior, pero esta vez en python.

Aunque Hadoop está desarrollado en java y como hemos visto la ejecución que hemos hecho es sobre un fichero .jar, hay librerías auxiliares de Hadoop que te permiten implementar MapReduce en diferentes lenguajes y gracias a su ayuda, permitir a los desarrolladores trabajar con lenguajes más amigables o que les sean más familiares.

En este caso para poder implementar MapReduce en Python me apoyaré en Hadoop-Streaming.

Para implementar en Python el MapReduce hay que crear dos ficheros .py, uno con el código que se ejecutará en el mapper y otro con el código que se ejecutará en el reducer.

Voy a comenzar escribiendo el código del mapper en un fichero llamado `mapper.py`.

##Mapper

In [ ]:
%%writefile -a mapper.py

#Importación de las librerías necesarias
import sys
import io

input_stream = io.TextIOWrapper(sys.stdin.buffer, encoding='latin1')
for line in input_stream:
  line = line.strip() # Se eliminan posibles espacios sobrantes antes y después

  # Si la linea está vacía, paso a la siguiente
  if not line:
    continue

  line = line.lower() # Se pasa el contenido de la linea a minúsculas

  values = line.split(',') # Separo el contenido de mi linea por ',' ya que es el delimitador de las columnas
  country = values[0].strip().strip('"') # Obtengo el nombre del pais (primer elemento del índice) omito espacios y comillas dobles al inicio y final del país

  # Evito la cabecera Entity para que no la contemple como país
  if country == "entity":
    continue

  print(f"{country.title()}\t{1}") # Pinto en el stdout el contenido que me interesa con el formato necesario por el Reducer para HadoopStreaming (clave y valor separados por una tabulación)

Writing mapper.py


El código del maper es bastante sencillo. Para integrarlo con HadoopStreaming partimos de que el contenido que va a recibir el mapper vendrá por el stdin y hay que devolver el resultado en el stdout(pintarlo con print) con el formato clave - valor separadas por tabulación.

Lo primero que hacemos es importar las librerías necesarias y leer el contenido del stdin. Lo leemos con la codificación `latin1` para evitar problemas con caracteres con tildes o caracteres raros.

Una vez leido el contenido del map, nos recorremos todas las lineas proporcionadas. Primero eliminamos posibles espacios al inicio y al final de cada linea (viene bien, aunque mi dataset no tiene estos caracteres sobrantes). Si la linea viene vacía, paso a la siguiente. Después paso el contenido de la linea a minúsculas, así unifico que un país aparezca de varias formas como Spain, Spain o SPAIN.

Después de realizar las transformaciones pertinentes separo el conjunto de la fila en cada uno de sus valores separados por `,`, caracter que es el delimitador en mi dataset. Como solamente me interesa el país, que está ubicado en la primera columna, me quedo con el primer elemento. Al país le elimino los espacios y las comillas que pueda tener antes y después para unificar los mismos paises escritos en formatos diferentes.

Una vez hecho todo imprimo (saldrá por stdout) el nombre del país pasándole la primera letra a mayúscula de cada palabra y el valor 1 separados por un tabulador. El formato sería `<country><TAB>1 `. Este formato es importante, ya que es el que necesita HadoopStreaming.

##Reducer

Una vez implementado el mapper, voy a pasar a implementar y explicar el Reducer.

In [ ]:
%%writefile -a reducer.py

# Importación de las librerías necesarias
import sys

current_country = None
current_count = 0

# Recorro todas las lineas proporcionadas en el stdin por HadoopStreaming
for line in sys.stdin:
    # Si la linea está vacía pasamos a la siguiente
    if not line:
      continue

    # Se eliminan posibles espacios sobrantes antes y después
    line = line.strip()

    # Separamos la linea por tabulador en el pais y las apariciones. Definimos
    # que como mucho haga una separación
    country, count = line.split('\t', 1)
    try:
      count = int(count)
    except ValueError:
      # Si el número, realmente no es un número, pasamos al siguiente país
      continue

    # Cuento las ocurrencias hasta que cambiemos de país, no uso diccionario
    # porque Hadoop ordena las claves alfabéticamente
    if current_country == country:  # Sigo en el mismo país que la iteración anterior
        current_count += count
    else: # He cambiado al siguiente país
        if current_country is not None:
            # Escribo los resultados del país anterior si tiene valor
            print(f"{current_country}\t{current_count}")
        # Actualizo el contador de apariciones y el país al actual
        current_count = count
        current_country = country

# Pinto el último resultado si es necesario
if current_country is not None:
  print(f"{current_country}\t{current_count}")

Writing reducer.py


El reducer recibe una lista de paises con su número de apariciones en cada linea, que previamente ha ordenado alfabéticamente Hadoop. Lo que recibe tendrá el formato clave-valor separados por una tabulación, donde la clave es el país y el valor el número de apariciones (tal como está hecho el maper solo tendrá un 1 por cada fila).

Este reducer se recorre todas las entradas y va sumando el número obtenido por cada pais a un contador. Cuando cambia de país (al enviarlos Hadoop ordenados) se considera que no va a volver a aparecer el país anterior, por lo que se pinta el total de apariciones y el nombre del país. Se sigue este flujo hasta finalizar las lineas.


##Ejecución de Hadoop-Streaming

Hadoop-Streaming nos va a permitir ejecutar nuestro mapper y reducer en python. Este recurso no hay que descargarlo a parte, viene por defecto con la descarga de Hadoop. Se ubica en `share/hadoop/tools/sources`, como nosotros hemos reubicado nuestra instalación de Hadoop a `/usr/local/hadoop` debería ubicarse en la ruta `/usr/local/hadoop/share/hadoop/tools/lib`.

Antes de seguir voy a buscar que realmente esté descargado.

In [ ]:
!hdfs dfs -ls /usr/local/hadoop/share/hadoop/tools/lib/ | grep hadoop-streaming

-rw-r--r--   1 1001 1001     141797 2025-08-20 10:54 /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.2.jar


Como podemos ver está en la ruta comentada anteriormente. En caso de no encontrarlo o desconocer la ruta, se puede utilizar el comando `!find / -name 'hadoop-streaming*.jar'` para revisar en todo el servidor donde se ubica.

Una vez he comprobado que lo tengo instalado, solamente queda su ejecución. El comando para ejecutarlo necesita que le indiquemos varias opciones. Voy a escribir el comando y paso a explicar cada una de ellas:



```
hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.2.jar
   -input datasets
   -output output
   -file mapper.py  -file reducer.py
   -mapper 'python mapper.py'
   -reducer 'python reducer.py'
```



*   `hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.2.jar`: Esta linea le indica a Hadoop que ejecute un job utilizando la librería hadoop-streaming-3.4.2.jar. Por lo que le tenemos que indicar donde se ubica la librería a ejecutar.
*   `-input 20news-18828/rec.sport.baseball/100521`: Con la opción input le indicamos sobre que carpeta o fichero queremos que realice el MapReduce.
*   `-output CountryCounter_HadoopStreaming`: Con la opción output le indicamos la ruta donde queremos que aloje los resultados del MapReduce.
*   `-file mapper.py  -file reducer.py`: Con la opción file le indicamos que copie estos ficheros de local al nodo donde se ejecuta hadoop. De esta forma los tendrá disponibles. Si no hacemos esto es posible que hadoop no tenga acceso a los ficheros.
*   `-mapper 'python mapper.py'`: Define el comando que debe ejecutar el mapper. Hadoop se encargará de pasar cada linea de nuestro fichero al stdin de la ejecución de este comando.
*   `-reducer 'python reducer.py'`: Define el comando que debe ejecutar el reducer.

Ahora voy a proceder a ejecutar el comando y ver los resultados.



In [ ]:
!hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.2.jar \
   -input datasets/ \
   -output CountryCounter_HadoopStreaming \
   -file mapper.py  -file reducer.py \
   -mapper 'python mapper.py' \
   -reducer 'python reducer.py'

2026-01-13 15:10:34,436 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [mapper.py, reducer.py] [] /tmp/streamjob6865207218168283143.jar tmpDir=null
2026-01-13 15:10:35,587 WARN impl.MetricsSystemImpl: JobTracker metrics system already initialized!
2026-01-13 15:10:35,864 INFO mapred.FileInputFormat: Total input files to process : 2
2026-01-13 15:10:35,893 INFO mapreduce.JobSubmitter: number of splits:2
2026-01-13 15:10:36,232 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_local972250247_0001
2026-01-13 15:10:36,236 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-01-13 15:10:36,645 INFO mapred.LocalDistributedCacheManager: Localized file:/content/mapper.py as file:/tmp/hadoop-root/mapred/local/job_local972250247_0001_3c5e57bd-a524-41ad-8e7d-1e5f0b1eaf73/mapper.py
2026-01-13 15:10:36,686 INFO mapred.LocalDistributedCacheManager: Localized file:/content/reducer.py as file:/tmp/hadoop-root/mapred/local

Parece que el comando ha funcionado correctamente. Como le hemos indicado en la opción `output` que la carpeta donde queremos alojar los resultados es `CountryCounter_HadoopStreaming` voy a verificar primero que existe la carpeta y su contenido.

In [ ]:
!hdfs dfs -ls CountryCounter_HadoopStreaming/

Found 2 items
-rw-r--r--   1 root root          0 2026-01-13 15:10 CountryCounter_HadoopStreaming/_SUCCESS
-rw-r--r--   1 root root       2733 2026-01-13 15:10 CountryCounter_HadoopStreaming/part-00000


Como podemos ver con el ls, existe la carpeta indicada y tiene contenido. Los resultados indican que se ha ejecutado sin fallar ya que tenemos el fichero `_SUCCESS`. Vamos a ver el contenido del fichero `part-00000` para confirmar que realmente se ha hecho el proceso tal cual teníamos previsto.

In [ ]:
!hdfs dfs -cat CountryCounter_HadoopStreaming/part-00000

Albania	15
American Samoa	23
Andorra	23
Angola	42
Anguilla	27
Antigua And Barbuda	27
Argentina	36
Armenia	27
Aruba	27
Australia	26
Austria	37
Azerbaijan	19
Bahamas	27
Bahrain	30
Bangladesh	27
Barbados	27
Belarus	24
Belgium	29
Belize	34
Benin	26
Bermuda	27
Bhutan	27
Bolivia	27
Bonaire Sint Eustatius And Saba	16
Bosnia And Herzegovina	36
Botswana	24
Brazil	42
British Virgin Islands	27
Brunei	18
Bulgaria	27
Burkina Faso	27
Burundi	23
Cambodia	27
Cameroon	20
Canada	27
Cape Verde	27
Cayman Islands	27
Central African Republic	26
Chad	27
Chile	40
China	26
Colombia	27
Comoros	30
Congo	30
Cook Islands	27
Costa Rica	36
Cote D'Ivoire	4
Croatia	43
Cuba	27
Curacao	27
Cyprus	42
Czechia	44
Democratic Republic Of Congo	22
Denmark	42
Djibouti	25
Dominica	27
Dominican Republic	32
East Timor	16
Ecuador	27
Egypt	31
El Salvador	42
Estonia	43
Eswatini	29
Ethiopia	27
Fiji	27
Finland	38
France	42
French Guiana	10
French Polynesia	40
Gabon	11
Gambia	27
Georgia	30
Germany	39
Ghana	27
Greece	47
Grenada	27
Guadel

Vemos que el contenido del fichero era lo esperado, un listado de paises con su número de apariciones total entre los dos datasets ubicados en la carpeta `datasets` (Remarco que hemos podido hacerlo con dos datasets porque ambos seguían el mismo formato, con este mismo código si uno de los dos tuviese un formato diferente no funcionaría y habría que hacerlo de uno en uno o modificar el código). El resultado podemos ver que es el mismo que arrojó el MapReducer programado en Java.

Con esto podemos concluir que la ejecución del MapReducer programado en Python a través de la librería HadoopStreaming se ha realizado correctamente.